<a href="https://colab.research.google.com/github/AlexitoFernandez/practicas-google-colab/blob/unidad-3/Practica_6_Suport_Vector_Machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#"Machine Learning"
##Unidad III
### **Practica 6 - Suport Vector Machine**

Alumno: Jorge Alejandro Fernández De Los Santos.

Facilitador: José Gabriel Rodríguez Rivas.


In [2]:
#Cargar librerías
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_curve, auc, make_scorer, recall_score, log_loss
import gdown

# Cargar datos
file_id = '1HBU_efbbPg378pZvLZ1AtvRFAKrEFsob'
output = 'lending_club_2007_2011_6_states.csv'
gdown.download(id=file_id, output=output, quiet=False)
prestamos_df= pd.read_csv(output)
print(prestamos_df.head())

Downloading...
From: https://drive.google.com/uc?id=1HBU_efbbPg378pZvLZ1AtvRFAKrEFsob
To: /content/lending_club_2007_2011_6_states.csv
100%|██████████| 7.01M/7.01M [00:00<00:00, 45.0MB/s]


   loan_amnt  funded_amnt  funded_amnt_inv       term  int_rate  installment  \
0       2400         2400           2400.0  36 months     15.96        84.33   
1      10000        10000          10000.0  36 months     13.49       339.31   
2       3000         3000           3000.0  36 months     18.64       109.43   
3       5600         5600           5600.0  60 months     21.28       152.39   
4       5375         5375           5350.0  60 months     12.69       121.45   

  grade sub_grade            emp_title emp_length  ... application_type  \
0     C        C5                  NaN  10+ years  ...       Individual   
1     C        C1  AIR RESOURCES BOARD  10+ years  ...       Individual   
2     E        E1      MKC Accounting     9 years  ...       Individual   
3     F        F2                  NaN    4 years  ...       Individual   
4     B        B5            Starbucks   < 1 year  ...       Individual   

   acc_now_delinq chargeoff_within_12_mths delinq_amnt pub_rec_bankr

In [3]:
#Transformación de datos
if 'grade' in prestamos_df.columns: prestamos_df['grade_code'] = prestamos_df['grade'].factorize()[0]
if 'purpose' in prestamos_df.columns: prestamos_df['purpose_code'] = prestamos_df['purpose'].factorize()[0]
if 'addr_state' in prestamos_df.columns: prestamos_df['addr_state_code'] = prestamos_df['addr_state'].factorize()[0]
if 'home_ownership' in prestamos_df.columns: prestamos_df['home_ownership_code'] = prestamos_df['home_ownership'].factorize()[0]
print("Datos transformados.")

#Los meses al estar en stack, se transforman de texto a número y después se dividen en 12 para transformarlos a años
prestamos_df['loan_term_year'] = prestamos_df['term'].str.extract(r'(\d+)').astype(int) / 12

#La Regresión Logística es un clasificador binario, por lo que necesita convertir estas categorías en algo que pueda medir
prestamos_df['repaid'] = prestamos_df['loan_status'].apply(lambda x: 1 if x == 'Fully Paid' else 0)

# Selección de variables predictoras
X = prestamos_df[['funded_amnt', 'loan_term_year', 'int_rate', 'grade_code',
                  'purpose_code', 'addr_state_code', 'home_ownership_code',
                  'annual_inc', 'dti', 'revol_util', 'pub_rec_bankruptcies']]

# Variable objetivo o variable a predecir
y = prestamos_df["repaid"]

Datos transformados.


In [4]:
# Dividimos el dataFrame
# stratify mantiene la misma proporción de clases en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size= 0.4, stratify=y )

In [5]:
# verificamos la cantidad de registros asignados al dataframe de entrenamiento
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((11944, 11), (7964, 11), (11944,), (7964,))

In [6]:
# Escalamiento de variables en SVM
# --- Normalización de variables --
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
# --- Definimos el modelo SVM --
svm_model = SVC(
kernel='rbf',
# radial basis function (no lineal)
C=1.0,
# penalización por errores
gamma='scale',
# ajuste automático de gamma
class_weight='balanced',  # útil si hay desbalance
probability=True,
# permite obtener probabilidades
random_state=42
)

# --- Preprocesamiento de NaN antes del entrenamiento ---
# Identificar filas con NaN en X_train_scaled
nan_rows_mask = np.isnan(X_train_scaled).any(axis=1)

# Filtrar X_train_scaled y y_train para eliminar las filas con NaN
X_train_scaled_cleaned = X_train_scaled[~nan_rows_mask]
y_train_cleaned = y_train[~nan_rows_mask]

# --- Entrenamiento --
svm_model.fit(X_train_scaled_cleaned, y_train_cleaned)

SVC(class_weight='balanced', probability=True, random_state=42)

In [10]:
# Fase de prueba y Reporte de clasificación

# Identificar filas con NaN en X_test_scaled
nan_rows_mask_test = np.isnan(X_test_scaled).any(axis=1)

# Filtrar X_test_scaled y y_test para eliminar las filas con NaN
X_test_scaled_cleaned = X_test_scaled[~nan_rows_mask_test]
y_test_cleaned = y_test[~nan_rows_mask_test]

# --- Predicción --
y_pred_svm = svm_model.predict(X_test_scaled_cleaned)
print("Reporte de Clasificación (SVM):")
print(classification_report(y_test_cleaned, y_pred_svm, target_names=["No Pagado", "Pagado"]))
print("Matriz de Confusión (SVM):")
print(confusion_matrix(y_test_cleaned, y_pred_svm))

Reporte de Clasificación (SVM):
              precision    recall  f1-score   support

   No Pagado       0.23      0.58      0.33      1148
      Pagado       0.90      0.67      0.77      6683

    accuracy                           0.65      7831
   macro avg       0.57      0.62      0.55      7831
weighted avg       0.80      0.65      0.70      7831

Matriz de Confusión (SVM):
[[ 667  481]
 [2238 4445]]


In [13]:
# Análisis AUC-ROC de Probabilidades de la clase positiva
##  Probabilidades de la clase positiva ##
from sklearn.metrics import roc_curve, roc_auc_score
import plotly.graph_objects as go
# --- Probabilidades de la clase positiva ("Pagado" = 1) --
y_prob_svm = svm_model.predict_proba(X_test_scaled_cleaned)[:, 1]
# --- Calcular AUC-ROC --
auc = roc_auc_score(y_test_cleaned, y_prob_svm)
print(f"\n AUC-ROC: {auc:.3f}")
# --- Curva ROC --
fpr, tpr, thresholds = roc_curve(y_test_cleaned, y_prob_svm)
# --- Gráfica interactiva con Plotly --
fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name='SVM', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Aleatorio', line=dict(dash='dash')))
fig.update_layout(
title=f'Probabilidades de la clase positiva - Curva ROC (AUC = {auc:.3f})',
xaxis_title='Tasa de Falsos Positivos (1 - Especificidad)',
yaxis_title='Tasa de Verdaderos Positivos (Sensibilidad)',
width=700, height=500
)
fig.show()


 AUC-ROC: 0.670


In [15]:
#Análisis AUC-ROC de Probabilidades de la clase negativa
# --- Probabilidades de la clase negativa ("Pagado" = 0) --
y_prob_svm = svm_model.predict_proba(X_test_scaled_cleaned)[:, 0]
# --- Calcular AUC-ROC --
auc = roc_auc_score(y_test_cleaned, y_prob_svm)
print(f"\nAUC-ROC: {auc:.3f}")
# --- Curva ROC --
fpr, tpr, thresholds = roc_curve(y_test_cleaned, y_prob_svm)
# --- Gráfica interactiva con Plotly --
fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name='SVM', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Aleatorio', line=dict(dash='dash')))
fig.update_layout(
title=f'Probabilidades de la clase negativa - Curva ROC (AUC = {auc:.3f})',
xaxis_title='Tasa de Falsos Positivos (1 - Especificidad)',
yaxis_title='Tasa de Verdaderos Positivos (Sensibilidad)',
width=700, height=500
)
fig.show()


AUC-ROC: 0.330


In [17]:
from sklearn.metrics import roc_curve, roc_auc_score
import plotly.graph_objects as go
# --- Probabilidades predichas por el modelo --
# Asegúrate de tener esto antes:
# svm_model.fit(X_train_scaled, y_train)
# y_prob_svm = svm_model.predict_proba(X_test_scaled)
# Calcular probabilidades y AUC para ambas clases
y_prob = svm_model.predict_proba(X_test_scaled_cleaned)
# Clase positiva (Pagado = 1)
fpr_pos, tpr_pos, _ = roc_curve(y_test_cleaned, y_prob[:, 1], pos_label=1)
auc_pos = roc_auc_score(y_test_cleaned, y_prob[:, 1])
# Clase negativa (No Pagado = 0)
fpr_neg, tpr_neg, _ = roc_curve(y_test_cleaned, y_prob[:, 0], pos_label=0)
auc_neg = roc_auc_score(y_test_cleaned, y_prob[:, 0])
# --- Graficar con Plotly --
fig = go.Figure()
fig.add_trace(go.Scatter(
x=fpr_pos, y=tpr_pos,
mode='lines',
name=f'Clase positiva (Pagado) - AUC = {auc_pos:.3f}',
line=dict(color='blue', width=2)
))
fig.add_trace(go.Scatter(
x=fpr_neg, y=tpr_neg,
mode='lines',
name=f'Clase negativa (No Pagado) - AUC = {auc_neg:.3f}',
line=dict(color='green', width=2)
))
# Línea diagonal (modelo aleatorio)
fig.add_trace(go.Scatter(
x=[0, 1], y=[0, 1],
mode='lines',
name='Aleatorio',
line=dict(color='red', width=1, dash='dash')
))
# --- Personalización del gráfico --
fig.update_layout(
title="Comparación de Curvas ROC - Clases Positiva y Negativa",
xaxis_title="Tasa de Falsos Positivos (1 - Especificidad)",
yaxis_title="Tasa de Verdaderos Positivos (Sensibilidad)",
width=900, height=600,
legend=dict(x=0.6, y=0.05)
)
fig.show()

In [20]:
#Optimización de SVM enfocada en detectar creditos NO pagados
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, fbeta_score, classification_report, confusion_matrix
import numpy as np

# Definimos el F2-score como métrica personalizada (recall más importante que precisión)
f2_scorer = make_scorer(fbeta_score, beta=2, pos_label=0)

# Modelo base
svm = SVC(probability=False, class_weight='balanced', random_state=42)

# Definimos los hiperparámetros a explorar
param_grid = {
'C': [0.1, 1, 10],
'kernel': ['linear', 'rbf', 'poly'],
'gamma': ['scale', 'auto'],
'degree': [2, 3],  # solo afecta al kernel 'poly'
}

# Configuración del GridSearchCV
grid_search = GridSearchCV(
estimator=svm,
param_grid=param_grid,
scoring=f2_scorer,  # optimizamos el F2-score
cv=5,

# validación cruzada de 5 particiones
verbose=2,
n_jobs=-1
)

# usa todos los núcleos disponibles
# Entrenamiento
print("Entrenando GridSearchCV para SVM...\n")
grid_search.fit(X_train_scaled_cleaned, y_train_cleaned)

# Resultados
print("\n Búsqueda finalizada.")
print("Mejores hiperparámetros encontrados:")
print(grid_search.best_params_)
print("\nMejor puntaje promedio (F2-score):")
print(round(grid_search.best_score_, 4))

# Evaluación con los mejores parámetros
svm_optimo = grid_search.best_estimator_
y_pred_opt = svm_optimo.predict(X_test_scaled_cleaned)
print("\n Reporte de Clasificación (SVM optimizado para impagos):\n")
print(classification_report(y_test_cleaned, y_pred_opt, target_names=["No Pagado", "Pagado"]))
print("Matriz de Confusión:")
print(confusion_matrix(y_test_cleaned, y_pred_opt))

Entrenando GridSearchCV para SVM...

Fitting 5 folds for each of 36 candidates, totalling 180 fits

 Búsqueda finalizada.
Mejores hiperparámetros encontrados:
{'C': 0.1, 'degree': 2, 'gamma': 'scale', 'kernel': 'rbf'}

Mejor puntaje promedio (F2-score):
0.4681

 Reporte de Clasificación (SVM optimizado para impagos):

              precision    recall  f1-score   support

   No Pagado       0.21      0.64      0.32      1148
      Pagado       0.91      0.60      0.72      6683

    accuracy                           0.60      7831
   macro avg       0.56      0.62      0.52      7831
weighted avg       0.80      0.60      0.66      7831

Matriz de Confusión:
[[ 735  413]
 [2692 3991]]
